# Phase 1: Visualisation using AI


In [138]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def find_predictions():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        f = p / "data" / "processed" / "predictions.csv"
        if f.exists():
            return f
    raise FileNotFoundError("predictions.csv not found: run from inside the repo.")

df = pd.read_csv(find_predictions())
df["correct"] = df["correct"].astype(bool)
print(df.shape, "| languages:", df.locale.nunique(), "| intents:", df.intent.nunique())

(99060, 9) | languages: 52 | intents: 20


### Sanity checks

Before plotting, we check that the data is what we assume: the same utterances in every language, one true intent per utterance, and a consistent `correct` column.

In [139]:
print("rows per locale:", df.groupby("locale").size().unique())
print("one intent per id:", (df.groupby("id").intent.nunique() == 1).all())
print("correct == (pred == intent):", (df.correct == (df.pred == df.intent)).all())
print("overall accuracy %.3f | English %.3f" % (df.correct.mean(), df[df.locale == "en-US"].correct.mean()))

rows per locale: [1905]
one intent per id: True
correct == (pred == intent): True
overall accuracy 0.372 | English 0.585


### Number of utterances in each cell

In [140]:
n_per_intent = df[df.locale == "en-US"].intent.value_counts().sort_values()
print("utterances per intent (same in every language):")
print(n_per_intent.to_string())

utterances per intent (same in every language):
intent
iot_hue_lightoff          43
recommendation_events     43
lists_query               51
transport_query           51
lists_remove              52
qa_definition             57
play_podcasts             63
calendar_remove           67
cooking_recipe            72
play_radio                72
social_post               81
datetime_query            88
email_sendemail          114
email_query              119
news_query               124
calendar_query           126
qa_factoid               141
weather_query            156
play_music               176
calendar_set             209


### Accuracy per language and intent

Rows are languages sorted by their mean accuracy across intents. Columns are intents sorted by English accuracy. Green means high accuracy and red low.

In [141]:
ROW_SORT = "accuracy"     
COL_SORT = "english"       

acc = df.pivot_table(index="locale", columns="intent", values="correct", aggfunc="mean")
cnt = df.pivot_table(index="locale", columns="intent", values="correct", aggfunc="size")

row_order = acc.mean(axis=1).sort_values().index if ROW_SORT == "accuracy" else sorted(acc.index)
if COL_SORT == "english":   col_order = acc.loc["en-US"].sort_values().index
elif COL_SORT == "overall": col_order = acc.mean(axis=0).sort_values().index
else:                       col_order = sorted(acc.columns)

A, N = acc.loc[row_order, col_order], cnt.loc[row_order, col_order]
fig = go.Figure(go.Heatmap(
    z=A.values, x=list(A.columns), y=list(A.index), zmin=0, zmax=1, colorscale="RdYlGn",
    customdata=N.values, colorbar=dict(title="accuracy"),
    hovertemplate="locale %{y}<br>intent %{x}<br>accuracy %{z:.2f}<br>n = %{customdata}<extra></extra>"))
fig.update_layout(title="Accuracy per language × intent", height=900,
                  xaxis=dict(tickangle=45, title="intent"), yaxis=dict(title="locale", autorange="reversed"))
fig.show()

**Observations:**
1. A set of intents looks equally inaccurate accross all languages.
2. The remaining intents are distributed unevenly accross the langusages.
3. A group of languages performs worse than the rest across many intents.
 


## Accuracy per intent

Each dot is an intent. The x-axis is its accuracy in English and the y-axis its mean accuracy over the other 51 languages. The dashed diagonal marks "same accuracy as in English" and the bar shows the lowest to the highest language. The colour shows how often the model predicts the label compared with how often it truly occurs.

In [142]:
import plotly.express as px  
EN = "en-US"
others = [l for l in acc.index if l != EN]

       
true_share = df[df.locale == EN].intent.value_counts(normalize=True)
pred_share = df.pred.value_counts(normalize=True)

it = pd.DataFrame({
    "english": acc.loc[EN],
    "mean_other": acc.loc[others].mean(),
    "min_other": acc.loc[others].min(),
    "max_other": acc.loc[others].max(),
    "n_per_locale": cnt.loc[EN],
})
it["pred_ratio"] = pred_share.reindex(it.index) / true_share.reindex(it.index)

fig = go.Figure()
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="grey"))
fig.add_trace(go.Scatter(
    x=it.english, y=it.mean_other, mode="markers+text", text=it.index, textposition="top center",
    marker=dict(size=12, color=it.pred_ratio, colorscale="RdBu_r", cmid=1, colorbar=dict(title="predicted /<br>true share")),
    error_y=dict(type="data", symmetric=False, array=(it.max_other - it.mean_other), arrayminus=(it.mean_other - it.min_other)),
    customdata=np.c_[it.n_per_locale, it.min_other, it.max_other, it.pred_ratio],
    hovertemplate="%{text}<br>English %{x:.2f}<br>mean other %{y:.2f}<br>range %{customdata[1]:.2f} to %{customdata[2]:.2f}"
                  "<br>n per locale %{customdata[0]}<br>pred/true share %{customdata[3]:.2f}<extra></extra>"))
fig.update_layout(title="English accuracy vs accuracy in the other 51 languages (per intent)",
                  xaxis_title="English accuracy", yaxis_title="mean accuracy, other languages", height=750,
                  xaxis_range=[0, 1.05], yaxis_range=[-0.05, 1.0])
fig.show()
it.sort_values("mean_other").round(2)

,english,mean_other,min_other,max_other,n_per_locale,pred_ratio
intent,,,,,,
calendar_query,0.10,0.01,0.00,0.06,126,0.03
email_query,0.03,0.06,0.00,0.44,119,0.25
calendar_set,0.28,0.06,0.00,0.22,209,0.08
recommendation_events,0.26,0.10,0.00,0.35,43,1.01
datetime_query,0.36,0.20,0.01,0.62,88,0.48
transport_query,0.33,0.21,0.00,0.47,51,0.29
lists_query,0.69,0.24,0.02,0.51,51,0.65
play_radio,0.58,0.36,0.00,0.68,72,0.48
news_query,0.65,0.37,0.03,0.67,124,0.49


**Observation.**
- `calendar_query`, `calendar_set` and `email_query` are inaccurate in every language, so their failure is not specific to translation.
- Most other intents have high English accuracy, an average of around 0.5 in the other languages, and a spread that covers almost the whole range from near 0 to high accuracy. For these intents the language matters a great deal.
- Only two intents lie above the diagonal, `qa_factoid` and `email_query`. `email_query` is close to zero in both cases. For `qa_factoid` the model predicts the label more often than it occurs, so part of its accuracy in other languages may come from the model defaulting to it and not from recognizing it.

## Languages ranked on the intents English solves



To separate the language effect from intents that fail in English too, we only use the 13 intents with English accuracy of at least 0.5. Each dot is a language and the bar is the 95% interval of its accuracy. The cell below compares Japanese with the next best languages on the same utterances.

In [143]:
EN_THRESHOLD = 0.5

en_acc = acc.loc[EN]
solvable = en_acc[en_acc >= EN_THRESHOLD].index.tolist()
print(f"{len(solvable)} English-solvable intents:", sorted(solvable))

sub = df[df.intent.isin(solvable)]
g = sub.groupby("locale").correct.agg(acc="mean", n="size")
g["se"] = np.sqrt(g.acc * (1 - g.acc) / g.n)
g["rel_en"] = g.acc / g.loc[EN, "acc"]
g = g.sort_values("acc")

fig = go.Figure()
fig.add_trace(go.Scatter(x=g.acc, y=g.index, mode="markers", marker=dict(size=9),
                         error_x=dict(type="data", array=1.96 * g.se),
                         customdata=np.c_[g.rel_en, g.n],
                         hovertemplate="%{y}<br>accuracy %{x:.3f}<br>relative to English %{customdata[0]:.2f}<br>n=%{customdata[1]}<extra></extra>"))
fig.add_vline(x=g.loc[EN, "acc"], line_dash="dash", line_color="grey", annotation_text="English")
fig.update_layout(title="Accuracy on English-solvable intents, per language", height=950,
                  xaxis_title="accuracy", yaxis=dict(categoryorder="array", categoryarray=list(g.index)))
fig.show()

13 English-solvable intents: ['calendar_remove', 'cooking_recipe', 'email_sendemail', 'iot_hue_lightoff', 'lists_query', 'lists_remove', 'news_query', 'play_music', 'play_podcasts', 'play_radio', 'qa_definition', 'social_post', 'weather_query']


In [144]:
from scipy.stats import binomtest
C = sub.pivot(index="id", columns="locale", values="correct")
ranked = g.drop(EN).sort_values("acc", ascending=False).index[:5].tolist()
best = ranked[0]
print("accuracy:", {l: round(float(g.loc[l, "acc"]), 3) for l in ranked})
for other in ranked[1:]:
    b = int((C[best] & ~C[other]).sum()); c = int((~C[best] & C[other]).sum())
    print(f"{best} right and {other} wrong: {b} | {other} right and {best} wrong: {c} | sign test p = {binomtest(b, b + c, 0.5).pvalue:.4f}")

accuracy: {'ja-JP': 0.754, 'nl-NL': 0.714, 'fr-FR': 0.71, 'pt-PT': 0.706, 'it-IT': 0.704}
ja-JP right and nl-NL wrong: 147 | nl-NL right and ja-JP wrong: 102 | sign test p = 0.0052
ja-JP right and fr-FR wrong: 138 | fr-FR right and ja-JP wrong: 89 | sign test p = 0.0014
ja-JP right and pt-PT wrong: 147 | pt-PT right and ja-JP wrong: 93 | sign test p = 0.0006
ja-JP right and it-IT wrong: 157 | it-IT right and ja-JP wrong: 101 | sign test p = 0.0006


**Observation.**
- Japanese is clearly the best of the other languages. On the same utterances Japanese is right where these languages are wrong clearly more often than the reverse.
- The remaining languages decline gradually down to the weakest ones.

## Is there a break between strong and weak languages?


We look at the gaps between neighbouring languages in the ranking and at the distribution of accuracy across languages. Afterwards we check that the ranking does not depend on the 0.5 threshold used to define "English-solvable".

In [145]:
gaps = g.acc.diff().dropna().sort_values(ascending=False)
print("Largest gaps between neighbouring languages (accuracy points):")
print(gaps.head(5).round(3).to_string())
print("median gap: %.3f" % gaps.median())

fig = px.histogram(g.drop(EN), x="acc", nbins=20, title="Distribution of language accuracy (English-solvable intents, English excluded)")
fig.update_layout(xaxis_title="accuracy", yaxis_title="number of languages")
fig.show()

Largest gaps between neighbouring languages (accuracy points):
locale
en-US    0.060
is-IS    0.046
jv-ID    0.044
km-KH    0.041
ja-JP    0.040
median gap: 0.008


In [146]:
ranks = {}
for thr in (0.4, 0.5, 0.6, 0.7):
    sol = en_acc[en_acc >= thr].index
    ranks[f"thr {thr} ({len(sol)} intents)"] = df[df.intent.isin(sol)].groupby("locale").correct.mean()
ranks = pd.DataFrame(ranks).drop(EN)
ranks.corr(method="spearman").round(3)

,thr 0.4 (14 intents),thr 0.5 (13 intents),thr 0.6 (12 intents),thr 0.7 (9 intents)
thr 0.4 (14 intents),1.000,0.997,0.994,0.985
thr 0.5 (13 intents),0.997,1.000,0.997,0.989
thr 0.6 (12 intents),0.994,0.997,1.000,0.989
thr 0.7 (9 intents),0.985,0.989,0.989,1.000


**Observation.**
- The gaps between neighbouring languages are small.
- The histogram shows how many languages fall in each accuracy range. It is context and not the main result.
- The weakest languages are therefore the tail of a gradual decline and not a separate group. Any group size we choose is our own choice and has to be stated.
- The ranking hardly depends on the threshold.

## Script and family of the weakest and strongest languages

Each language is annotated with its script and its family. The tables count how many of the 10 weakest and 10 strongest languages fall into each script and family. The cell below the tables gives the base rate of Latin script.

In [147]:
META = {
 "af-ZA":("Latin","Germanic"), "am-ET":("Ethiopic","Semitic"), "ar-SA":("Arabic","Semitic"),
 "az-AZ":("Latin","Turkic"), "bn-BD":("Bengali","Indo-Aryan"), "ca-ES":("Latin","Romance"),
 "cy-GB":("Latin","Celtic"), "da-DK":("Latin","Germanic"), "de-DE":("Latin","Germanic"),
 "el-GR":("Greek","Hellenic"), "en-US":("Latin","Germanic"), "es-ES":("Latin","Romance"),
 "fa-IR":("Arabic","Indo-Iranian"), "fi-FI":("Latin","Uralic"), "fr-FR":("Latin","Romance"),
 "he-IL":("Hebrew","Semitic"), "hi-IN":("Devanagari","Indo-Aryan"), "hu-HU":("Latin","Uralic"),
 "hy-AM":("Armenian","Armenian"), "id-ID":("Latin","Austronesian"), "is-IS":("Latin","Germanic"),
 "it-IT":("Latin","Romance"), "ja-JP":("Japanese","Japonic"), "jv-ID":("Latin","Austronesian"),
 "ka-GE":("Georgian","Kartvelian"), "km-KH":("Khmer","Austroasiatic"), "kn-IN":("Kannada","Dravidian"),
 "ko-KR":("Hangul","Koreanic"), "lv-LV":("Latin","Baltic"), "ml-IN":("Malayalam","Dravidian"),
 "mn-MN":("Cyrillic","Mongolic"), "ms-MY":("Latin","Austronesian"), "my-MM":("Myanmar","Sino-Tibetan"),
 "nb-NO":("Latin","Germanic"), "nl-NL":("Latin","Germanic"), "pl-PL":("Latin","Slavic"),
 "pt-PT":("Latin","Romance"), "ro-RO":("Latin","Romance"), "ru-RU":("Cyrillic","Slavic"),
 "sl-SL":("Latin","Slavic"), "sq-AL":("Latin","Albanian"), "sv-SE":("Latin","Germanic"),
 "sw-KE":("Latin","Niger-Congo"), "ta-IN":("Tamil","Dravidian"), "te-IN":("Telugu","Dravidian"),
 "th-TH":("Thai","Kra-Dai"), "tl-PH":("Latin","Austronesian"), "tr-TR":("Latin","Turkic"),
 "ur-PK":("Arabic","Indo-Aryan"), "vi-VN":("Latin","Austroasiatic"), "zh-CN":("Han","Sino-Tibetan"),
 "zh-TW":("Han","Sino-Tibetan"),
}
meta = pd.DataFrame(META, index=["script", "family"]).T.rename_axis("locale")
assert set(df.locale) <= set(meta.index)

N_GROUP = 10
order = g.drop(EN).sort_values("acc").index.tolist()
bottom, top = order[:N_GROUP], order[-N_GROUP:]
print("weakest:", bottom)
print("strongest:", top)

comp = pd.DataFrame({
    "weakest": meta.loc[bottom].script.value_counts(),
    "strongest": meta.loc[top].script.value_counts(),
}).fillna(0).astype(int)
display(comp)
fam = pd.DataFrame({
    "weakest": meta.loc[bottom].family.value_counts(),
    "strongest": meta.loc[top].family.value_counts(),
}).fillna(0).astype(int)
display(fam)

weakest: ['am-ET', 'my-MM', 'ka-GE', 'km-KH', 'cy-GB', 'kn-IN', 'ml-IN', 'sw-KE', 'mn-MN', 'te-IN']
strongest: ['pl-PL', 'es-ES', 'de-DE', 'ru-RU', 'zh-CN', 'it-IT', 'pt-PT', 'fr-FR', 'nl-NL', 'ja-JP']


,weakest,strongest
script,,
Cyrillic,1,1
Ethiopic,1,0
Georgian,1,0
Han,0,1
Japanese,0,1
Kannada,1,0
Khmer,1,0
Latin,2,7
Malayalam,1,0


,weakest,strongest
family,,
Austroasiatic,1,0
Celtic,1,0
Dravidian,3,0
Germanic,0,2
Japonic,0,1
Kartvelian,1,0
Mongolic,1,0
Niger-Congo,1,0
Romance,0,4


**Observation.**
- 7 of the 10 strongest languages use the Latin script, against 2 of the 10 weakest.
- By family, the strongest contain four Romance and two Germanic languages and the weakest none. Three Dravidian languages are among the weakest and none among the strongest. Otherwise the weakest group is diverse.


## Accuracy by script

Each dot is a language on the English-solvable intents, grouped by script and coloured by family.

In [148]:
gm = g.drop(EN).join(meta).reset_index()
fig = px.strip(gm, x="script", y="acc", color="family", hover_name="locale",
               title="Accuracy on English-solvable intents by script (colour = family)")
fig.update_traces(marker_size=11); fig.update_layout(height=600, yaxis_title="accuracy"); fig.show()

**Observation.**
- Latin-script languages range from 0.24 to 0.71 (median 0.58). Other scripts range from 0.19 to 0.75 (median 0.48). The ranges overlap almost completely.
- Welsh (0.24), Swahili (0.29) and Javanese (0.38) are weak despite the Latin script, while Japanese (0.75), Russian and Chinese (0.70), Korean (0.65) and Traditional Chinese (0.61) are strong with other scripts.
- Script is associated with the ranking but does not explain it. From outputs alone we cannot say why; candidates are training data, tokenization and similarity to English.

## Where do the errors go?

For the intents English solves, we count which label the model predicts when it is wrong, separately for the 10 weakest and the 10 strongest languages.

In [149]:
errs = sub[~sub.correct]
share = pd.DataFrame({
    "weakest": errs[errs.locale.isin(bottom)].pred.value_counts(normalize=True),
    "strongest": errs[errs.locale.isin(top)].pred.value_counts(normalize=True),
}).fillna(0).sort_values("weakest")

fig = go.Figure()
fig.add_bar(y=share.index, x=share.weakest, name="weakest languages", orientation="h")
fig.add_bar(y=share.index, x=share.strongest, name="strongest languages", orientation="h")
fig.update_layout(barmode="group", height=700, xaxis_title="share of errors",
                  title="Predicted label when the model is wrong (English-solvable intents)")
fig.show()

**Observation.**
- `qa_factoid` is the most common wrong label in both groups, followed by `social_post`.
- The three most common wrong labels cover 62% of the errors in the weakest languages and 51% in the strongest, so the weakest languages collapse more strongly into a few labels.
- The strongest languages spread their errors somewhat more, for example `lists_remove`.

## Which intents lose the most?

For each of the 13 English-solvable intents, the mean accuracy in English, in the 10 strongest and in the 10 weakest languages. The line connects the weakest group to English.

In [150]:
per_intent = pd.DataFrame({
    "English": acc.loc[EN, solvable],
    "strongest": acc.loc[top, solvable].mean(),
    "weakest": acc.loc[bottom, solvable].mean(),
}).sort_values("weakest")

fig = go.Figure()
for i, row in per_intent.iterrows():
    fig.add_shape(type="line", x0=row.weakest, x1=row.English, y0=i, y1=i, line=dict(color="lightgrey"))
for col, sym in (("English", "diamond"), ("strongest", "circle"), ("weakest", "circle-open")):
    fig.add_trace(go.Scatter(x=per_intent[col], y=per_intent.index, mode="markers", name=col, marker=dict(size=11, symbol=sym)))
fig.update_layout(title="Accuracy per English-solvable intent: English vs strongest vs weakest languages",
                  xaxis_title="accuracy", height=650, yaxis=dict(categoryorder="array", categoryarray=list(per_intent.index)))
fig.show()

**Observation.**
- On average the weakest languages lose 0.58 accuracy compared with English, the strongest only 0.12.
- The weakest languages are close to the floor on `lists_query`, `play_podcasts`, `play_radio` and `iot_hue_lightoff`, where English reaches 0.58 to 1.00. `social_post` keeps the most.
- Even the strongest languages lose a lot on `lists_query` (0.69 to 0.38) and `calendar_remove` (0.99 to 0.70).

## Summary of Phase I findings

1. **Two kinds of intents.** Some intents are inaccurate in every language, including English, so for them the behaviour is a task or label problem and not a language problem. For most other intents English works and accuracy in other languages varies widely.
2. **A gradual decline across languages, with a weak tail.** On the intents English solves, Japanese is clearly the best non-English language, and a set of languages reach only about a quarter of English accuracy. There is no clear break between strong and weak languages. Latin script is associated with higher accuracy but does not explain the ranking.
3. **Errors collapse into a few labels.** About a third of the wrong answers are `qa_factoid`, and the weakest languages collapse more strongly than the strongest. Accuracy alone can mislead here: a label such as `qa_factoid` gains accuracy when the model defaults to it.
